# 共享单车小时级需求预测

目标：使用 `hour.csv` 预测某个小时的共享单车总租赁量 `cnt`。

与日级模型不同，这里增加了 `hr`（小时）特征；`casual` 与 `registered` 不作为特征，因为两者相加正好等于目标 `cnt`，使用它们会造成数据泄漏。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

data_path = Path('../data/hour.csv')
if not data_path.exists():
    raise FileNotFoundError('找不到 hour.csv。请确认在 notebooks 文件夹中打开此 Notebook。')

df_hour = pd.read_csv(data_path, parse_dates=['dteday'])
df_hour.head()

## 1. 了解小时级数据

先确认样本量、字段类型和缺失值。每一行代表某一天的某一个小时。

In [ ]:
print(f'数据形状：{df_hour.shape}')
display(df_hour.info())
display(df_hour.isna().sum().to_frame('缺失值数量').T)
df_hour[['dteday', 'hr', 'cnt']].describe()

## 2. 探索不同时段的需求

观察一天中不同小时的平均租赁量，通常可看到早晚通勤高峰。

In [ ]:
hourly_mean = df_hour.groupby('hr', as_index=False)['cnt'].mean()
plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly_mean, x='hr', y='cnt', marker='o')
plt.xticks(range(24))
plt.title('Average Rental Count by Hour')
plt.xlabel('Hour')
plt.ylabel('Average Rental Count')
plt.show()

hourly_mean.sort_values('cnt', ascending=False).head(5)

## 3. 按时间划分训练集和测试集

不能随机打乱数据。前 80% 的历史小时用于训练，后 20% 用于测试，模拟“用过去预测未来”。

In [ ]:
FEATURES_HOURLY = [
    'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
    'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
]
TARGET = 'cnt'

df_hour = df_hour.sort_values(['dteday', 'hr']).reset_index(drop=True)
split_index = int(len(df_hour) * 0.8)
train = df_hour.iloc[:split_index]
test = df_hour.iloc[split_index:]

X_train, y_train = train[FEATURES_HOURLY], train[TARGET]
X_test, y_test = test[FEATURES_HOURLY], test[TARGET]

print(f'训练集：{X_train.shape[0]} 条，时间范围：{train.dteday.min().date()} 至 {train.dteday.max().date()}')
print(f'测试集：{X_test.shape[0]} 条，时间范围：{test.dteday.min().date()} 至 {test.dteday.max().date()}')

## 4. 建立基线模型并比较

先用线性回归作为基线，再使用随机森林。评价指标仍为 MAE、RMSE 和 R²。

In [ ]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        '模型': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': root_mean_squared_error(y_test, pred),
        'R²': r2_score(y_test, pred),
    }, pred

linear_result, linear_pred = evaluate_model('线性回归', LinearRegression())
rf_result, rf_pred = evaluate_model(
    '随机森林（基线）',
    RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
)

results = pd.DataFrame([linear_result, rf_result]).set_index('模型')
results.round(3)

## 5. 查看随机森林误差与特征重要性

误差图用来发现模型在哪些时段预测偏差较大；特征重要性描述模型在整体训练中最常使用哪些特征，不等于因果关系。

In [ ]:
comparison = test[['dteday', 'hr', 'cnt']].copy()
comparison['prediction'] = rf_pred
comparison['absolute_error'] = (comparison['cnt'] - comparison['prediction']).abs()
display(comparison.head())

# 重新训练同配置模型，方便取得特征重要性。
rf_model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
importance = pd.Series(rf_model.feature_importances_, index=FEATURES_HOURLY).sort_values()
importance.plot.barh(figsize=(8, 5), title='Hourly Model Feature Importance')
plt.xlabel('Importance')
plt.show()

## 小结（运行后填写）

- 小时级需求的高峰时段：……
- 表现更好的模型：……，其 MAE / RMSE / R² 为：……
- 最重要的特征：……
- 下一步：对随机森林做时间序列交叉验证调参，再决定是否部署到网页。